# Aviation Accidents: Data Cleaning

**Client:** an airline / airplane insurer. **Question:** which airplane makes/models show low rates of destruction and low likelihood of fatal or serious injury, and which general conditions matter?

**Data:** NTSB-style aviation accident records, 1948-2023 (88,889 rows). **This notebook** loads, inspects and cleans the data and builds the two key measures (fatal/serious injury fraction and destroyed flag). It writes `data/cleaned_aviation.csv` for the analysis notebook.

**Client constraints we enforce here**
- Airplanes only (no helicopters, gliders, balloons, ...)
- Professionally built only (`Amateur.Built == 'No'`)
- Events from 1983 onward (assumed 40-year max service life of a make/model)
- Makes with at least 50 records

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

## 1. Data Loading and Inspection

In [2]:
df = pd.read_csv('data/AviationData.csv', encoding='latin-1', low_memory=False)
print(df.shape)
df.head()

(88889, 31)


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,Injury.Severity,Aircraft.damage,Aircraft.Category,Registration.Number,Make,Model,Amateur.Built,Number.of.Engines,Engine.Type,FAR.Description,Schedule,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,Fatal(2),Destroyed,NaN,NC6404,Stinson,108-3,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,Fatal(4),Destroyed,NaN,N5069P,Piper,PA24-180,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,Fatal(3),Destroyed,NaN,N5142R,Cessna,172M,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,Fatal(2),Destroyed,NaN,N1168J,Rockwell,112,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,Fatal(1),Destroyed,NaN,N15NY,Cessna,501,No,NaN,NaN,NaN,NaN,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [3]:
# Datatypes and non-null counts
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  str    
 7   Longitude               34373 non-null  str    
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-null  str    
 

In [4]:
# Share of missing values per column
(df.isna().mean().sort_values(ascending=False) * 100).round(1)

Schedule                  85.8
Air.carrier               81.3
FAR.Description           64.0
Aircraft.Category         63.7
Longitude                 61.3
Latitude                  61.3
Airport.Code              43.6
Airport.Name              40.7
Broad.phase.of.flight     30.6
Publication.Date          15.5
Total.Serious.Injuries    14.1
Total.Minor.Injuries      13.4
Total.Fatal.Injuries      12.8
Engine.Type                8.0
Report.Status              7.2
Purpose.of.flight          7.0
Number.of.Engines          6.8
Total.Uninjured            6.7
Weather.Condition          5.1
Aircraft.damage            3.6
Registration.Number        1.6
Injury.Severity            1.1
Country                    0.3
Amateur.Built              0.1
Model                      0.1
Make                       0.1
Location                   0.1
Accident.Number            0.0
Investigation.Type         0.0
Event.Id                   0.0
Event.Date                 0.0
dtype: float64

In [5]:
df.describe()

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


**Observations**
- `Event.Date` is stored as a string; we convert it to datetime.
- `Aircraft.Category` is missing for about 64% of rows, mostly older records. Dropping all of them would throw away a lot of airplane data, so we try to recover the category from other records of the same make/model.
- Injury counts and `Aircraft.damage` (note the lowercase `d`) have some NaNs.
- Several categorical columns use placeholders (`Unknown`, `UNK`, `Unk`, `NONE`) for missing values.
- Injury counts have extreme maxima (hundreds), which are real large-airliner accidents and not errors.

## 2. Filtering aircraft and events

In [6]:
df['Event.Date'] = pd.to_datetime(df['Event.Date'])
print(df['Event.Date'].min(), df['Event.Date'].max())
df['Aircraft.Category'].value_counts(dropna=False)

1948-10-24 00:00:00 2022-12-29 00:00:00


Aircraft.Category
NaN                  56602
Airplane             27617
Helicopter            3440
Glider                 508
Balloon                231
Gyrocraft              173
Weight-Shift           161
Powered Parachute       91
Ultralight              30
Unknown                 14
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
ULTR                     1
Name: count, dtype: int64

### Recovering missing `Aircraft.Category`
**Assumption:** if a (Make, Model) pair appears elsewhere in the data with a known category, a record with the same pair and a missing category is that category. We use the most common known category for each pair. This is only done where a pair has a known category; everything else stays NaN and is dropped by the Airplane filter (a conservative choice).

In [7]:
# Normalise the key used for the lookup (uppercase, stripped)
df['_make_key'] = df['Make'].str.upper().str.strip()
df['_model_key'] = df['Model'].str.upper().str.strip()

known = df[df['Aircraft.Category'].notna() & ~df['Aircraft.Category'].isin(['Unknown', 'UNK'])]
lookup = (known.groupby(['_make_key', '_model_key'])['Aircraft.Category']
               .agg(lambda s: s.mode().iloc[0]))

before = df['Aircraft.Category'].isna().sum()
keys = pd.MultiIndex.from_frame(df[['_make_key', '_model_key']])
imputed = pd.Series(lookup.reindex(keys).values, index=df.index)
df['Aircraft.Category'] = df['Aircraft.Category'].fillna(imputed)
print(f"Missing category: {before} -> {df['Aircraft.Category'].isna().sum()}")
df['Aircraft.Category'].value_counts().head(5)

Missing category: 56602 -> 11909


Aircraft.Category
Airplane      67958
Helicopter     7241
Glider          881
Balloon         376
Gyrocraft       200
Name: count, dtype: int64

In [8]:
# Amateur.Built: NaN for ~100 rows. We cannot assume they are professional builds, so they are dropped.
print(df['Amateur.Built'].value_counts(dropna=False))

n0 = len(df)
df = df[df['Aircraft.Category'] == 'Airplane']
print('After airplane filter:', len(df), f'(removed {n0-len(df)})')
n0 = len(df)
df = df[df['Amateur.Built'] == 'No']
print('After professional-build filter:', len(df), f'(removed {n0-len(df)})')
n0 = len(df)
df = df[df['Event.Date'] >= '1983-01-01']
print('After 1983+ filter:', len(df), f'(removed {n0-len(df)})')
df = df.copy()

Amateur.Built
No     80312
Yes     8475
NaN      102
Name: count, dtype: int64
After airplane filter: 67958 (removed 20931)
After professional-build filter: 64379 (removed 3579)
After 1983+ filter: 61384 (removed 2995)


## 3. Key measurables

### 3a. Fatal / serious injury fraction
- **Passengers (persons on board)** = fatal + serious + minor + uninjured. The data does not separate crew from passengers, so this counts everyone aboard.
- **Missing counts:** if at least one of the four injury columns is recorded, we treat the other missing ones as 0 (the NTSB leaves a count blank when it is zero). If all four are missing, the row has no usable outcome and is dropped.
- Rows with 0 persons on board are dropped (the fraction is undefined).
- **`fatal_serious_frac` = (fatal + serious) / total persons on board.**

In [9]:
inj = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']
print(df[inj].isna().sum())
all_missing = df[inj].isna().all(axis=1)
print('Rows with all four missing:', all_missing.sum())

df = df[~all_missing].copy()
df[inj] = df[inj].fillna(0)

df['total_passengers'] = df[inj].sum(axis=1)
df['fatal_serious'] = df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']
print('Rows with 0 persons aboard:', (df['total_passengers'] == 0).sum())
df = df[df['total_passengers'] > 0].copy()

df['fatal_serious_frac'] = df['fatal_serious'] / df['total_passengers']
print(df['fatal_serious_frac'].describe())
df['fatal_serious_frac'].between(0, 1).all()

Total.Fatal.Injuries      7875
Total.Serious.Injuries    8652
Total.Minor.Injuries      8180
Total.Uninjured           3618
dtype: int64
Rows with all four missing: 114


Rows with 0 persons aboard: 958
count    60312.000000
mean         0.269143
std          0.428012
min          0.000000
25%          0.000000
50%          0.000000
75%          0.666667
max          1.000000
Name: fatal_serious_frac, dtype: float64


np.True_

### 3b. Aircraft damage / destroyed flag
Values are `Substantial`, `Destroyed`, `Minor`, `Unknown` and NaN. `Unknown` and NaN carry no information on destruction, so those rows are dropped. **`destroyed`** = 1 if damage is `Destroyed`, else 0.

In [10]:
print(df['Aircraft.damage'].value_counts(dropna=False))
df = df[df['Aircraft.damage'].isin(['Destroyed', 'Substantial', 'Minor'])].copy()
df['destroyed'] = (df['Aircraft.damage'] == 'Destroyed').astype(int)
df['destroyed'].mean()

Aircraft.damage
Substantial    45141
Destroyed      11954
NaN             1587
Minor           1556
Unknown           74
Name: count, dtype: int64


np.float64(0.20381579171710626)

## 4. Cleaning the `Make` column
**Cleaning tasks**
1. Missing Make: drop rows.
2. Standardise case and whitespace (uppercase, strip, collapse repeated spaces).
3. Remove punctuation and corporate suffixes (`INC`, `CORP`, `CO`, `LTD`, `COMPANY`, ...).
4. Merge known spelling variants of the same manufacturer (e.g. `CIRRUS DESIGN CORP` / `CIRRUS`, `DEHAVILLAND` / `DE HAVILLAND`, `AIR TRACTOR INC` / `AIR TRACTOR`).
5. Keep makes with at least **50** records.

In [11]:
print('Unique makes before:', df['Make'].nunique())
df = df.dropna(subset=['Make']).copy()

make = (df['Make'].str.upper()
        .str.replace(r'[^\w\s]', ' ', regex=True)
        .str.replace(r'\b(INC|CORP|CORPORATION|CO|COMPANY|LTD|LLC|INTERNATIONAL|INTL|AIRCRAFT|AIRPLANE|AIRPLANES|DESIGN|MFG|MANUFACTURING)\b', ' ', regex=True)
        .str.replace(r'\s+', ' ', regex=True).str.strip())

# Look at the largest remaining groups for variants
make.value_counts().head(40)

Unique makes before: 1321


Make
CESSNA                  24851
PIPER                   13577
BEECH                    4498
MOONEY                   1232
BOEING                   1007
GRUMMAN                   965
BELLANCA                  915
AIR TRACTOR               850
AERONCA                   533
MAULE                     496
CHAMPION                  474
CIRRUS                    443
STINSON                   392
LUSCOMBE                  364
AERO COMMANDER            330
TAYLORCRAFT               320
ROCKWELL                  315
NORTH AMERICAN            294
DE HAVILLAND              279
AYRES                     230
AVIAT                     222
MCDONNELL DOUGLAS         215
GRUMMAN AMERICAN          185
SCHWEIZER                 140
ERCOUPE ENG RESEARCH      134
LAKE                      123
GRUMMAN SCHWEIZER         120
DOUGLAS                   119
EMBRAER                   118
LEARJET                   118
SOCATA                    100
WACO                       98
DEHAVILLAND                95
MITSU

In [12]:
# Manual merges after inspecting the top makes (and near-duplicates)
merge_map = {
    'DEHAVILLAND': 'DE HAVILLAND',
    'DE HAVILLAND CANADA': 'DE HAVILLAND',
    'MCDONNELL DOUGLAS': 'MCDONNELL DOUGLAS',
    'DOUGLAS': 'MCDONNELL DOUGLAS',
    'BEECHCRAFT': 'BEECH',
    'RAYTHEON': 'BEECH',
    'HAWKER BEECHCRAFT': 'BEECH',
    'MOONEY AIRPLANE': 'MOONEY',
    'AERO COMMANDER': 'AERO COMMANDER',
    'COMMANDER': 'AERO COMMANDER',
    'ROCKWELL': 'AERO COMMANDER',
    'GULFSTREAM AMERICAN': 'GRUMMAN',
    'AMERICAN CHAMPION': 'CHAMPION',
    'AIR TRACTOR': 'AIR TRACTOR',
    'EMBRAER': 'EMBRAER',
    'CESSNA AIRCRAFT': 'CESSNA',
    'PIPER AIRCRAFT': 'PIPER',
    'THE BOEING': 'BOEING',
    'BOEING OF CANADA': 'BOEING',
    'AIRBUS INDUSTRIE': 'AIRBUS',
    'CIRRUS': 'CIRRUS',
    'SOCATA': 'SOCATA',
    'GRUMMAN AMERICAN': 'GRUMMAN',
    'GRUMMAN SCHWEIZER': 'SCHWEIZER',
    'DIAMOND IND': 'DIAMOND',
    'ERCOUPE ENG RESEARCH': 'ERCOUPE',
    'GRUMMAN ACFT ENG COR SCHWEIZER': 'SCHWEIZER',
    'GRUMMAN AMERICAN AVN': 'GRUMMAN',
    'DIAMOND': 'DIAMOND',
}
df['Make'] = make.replace(merge_map)
print('Unique makes after cleaning:', df['Make'].nunique())

make_counts = df['Make'].value_counts()
keep = make_counts[make_counts >= 50].index
df = df[df['Make'].isin(keep)].copy()
print('Makes kept (>=50 records):', len(keep), '| rows:', len(df))
df['Make'].value_counts()

Unique makes after cleaning: 859
Makes kept (>=50 records): 44 | rows: 55810


Make
CESSNA                 24851
PIPER                  13577
BEECH                   4651
GRUMMAN                 1245
MOONEY                  1232
BOEING                  1007
BELLANCA                 915
AIR TRACTOR              850
AERO COMMANDER           654
CHAMPION                 542
AERONCA                  533
MAULE                    496
CIRRUS                   443
STINSON                  392
DE HAVILLAND             374
LUSCOMBE                 364
MCDONNELL DOUGLAS        334
TAYLORCRAFT              320
SCHWEIZER                318
NORTH AMERICAN           294
AYRES                    230
AVIAT                    222
ERCOUPE                  206
DIAMOND                  125
LAKE                     123
EMBRAER                  118
LEARJET                  118
SOCATA                   100
AIRBUS                    99
WACO                      98
MITSUBISHI                92
RYAN                      89
SWEARINGEN                87
BOMBARDIER                87
GLOBE    

> `BEECH`/`RAYTHEON` and `ROCKWELL`/`COMMANDER` merges reflect corporate lineage of the same product lines; if a client wants these separated, remove those entries from `merge_map`.

## 5. Cleaning the `Model` column
- Drop NaN models.
- Standardise formatting: uppercase and strip every non-alphanumeric character so that `PA-28-180`, `PA 28 180` and `PA28180` match.
- Model names are **not unique across makes** (e.g. `172`-style short numbers and `UNKNOWN`/`SPECIAL` labels can recur across manufacturers), so we create `plane_type = Make + ' ' + Model` as the unique identifier.
- Remove placeholder models.

In [13]:
df = df.dropna(subset=['Model']).copy()
df['Model'] = (df['Model'].str.upper()
               .str.replace(r'[^A-Z0-9]', '', regex=True))
df = df[~df['Model'].isin(['', 'UNK', 'UNKNOWN', 'NONE', 'NA', 'SPECIAL', 'OTHER'])].copy()

# Are model labels unique to a make?
mm = df.groupby('Model')['Make'].nunique()
print('Models shared by >1 make:', (mm > 1).sum(), 'of', len(mm))
mm[mm > 1].sort_values(ascending=False).head(10)

Models shared by >1 make: 70 of 1568


Model
500        3
NAVION     3
NAVIONA    3
S2B        3
60         3
100        2
320        2
24         2
23         2
200        2
Name: Make, dtype: int64

In [14]:
df['plane_type'] = df['Make'] + ' ' + df['Model']
print('Unique plane types:', df['plane_type'].nunique())
df['plane_type'].value_counts().head(15)

Unique plane types: 1643


plane_type
CESSNA 152       2211
CESSNA 172       1614
CESSNA 172N      1086
PIPER PA28140     924
CESSNA 172M       786
CESSNA 150        742
CESSNA 172P       668
PIPER PA18        662
PIPER PA18150     618
CESSNA 182        608
CESSNA 180        593
PIPER PA28180     592
PIPER PA28161     556
CESSNA 150M       550
PIPER PA28181     542
Name: count, dtype: int64

## 6. Cleaning other columns
Planned treatments:
- **Engine.Type**: `Unknown`, `UNK`, `NONE`, `LR` -> NaN. Types with very few records (`Electric`, `Hybrid Rocket`, ...) -> `Other`.
- **Weather.Condition**: `UNK`, `Unk`, `Unknown` -> NaN. Leaves VMC (visual) and IMC (instrument conditions).
- **Number.of.Engines**: 0 engines is impossible for a powered airplane (a placeholder) -> NaN; values above 4 are too rare and suspicious -> NaN.
- **Purpose.of.flight**: `Unknown` -> NaN; the `Public Aircraft - *` variants merge into `Public Aircraft`; categories with <1% of records -> `Other`.
- **Broad.phase.of.flight**: `Unknown`, `Other` -> NaN.

We do not impute these values; NaNs are simply excluded from analyses of that factor.

In [15]:
for c in ['Engine.Type', 'Weather.Condition', 'Number.of.Engines', 'Purpose.of.flight', 'Broad.phase.of.flight']:
    print('-----', c)
    print(df[c].value_counts(dropna=False))

----- Engine.Type
Engine.Type
Reciprocating      49183
NaN                 2904
Turbo Prop          2217
Turbo Fan            887
Unknown              390
Turbo Jet            202
Turbo Shaft           13
Geared Turbofan        1
UNK                    1
Name: count, dtype: int64
----- Weather.Condition
Weather.Condition
VMC    49082
IMC     4354
NaN     1732
UNK      490
Unk      140
Name: count, dtype: int64
----- Number.of.Engines
Number.of.Engines
1.0    46375
2.0     6948
NaN     2148
3.0      140
4.0      107
0.0       80
Name: count, dtype: int64
----- Purpose.of.flight
Purpose.of.flight
Personal                     33204
Instructional                 7834
Aerial Application            3155
Unknown                       3079
Business                      2676
NaN                           2555
Positioning                    893
Other Work Use                 489
Ferry                          479
Aerial Observation             369
Executive/corporate            275
Public Aircra

In [16]:
# Engine type
df['Engine.Type'] = df['Engine.Type'].replace({'Unknown': np.nan, 'UNK': np.nan, 'NONE': np.nan, 'LR': np.nan})
rare = df['Engine.Type'].value_counts()
rare = rare[rare < 30].index
df['Engine.Type'] = df['Engine.Type'].replace({r: 'Other' for r in rare})

# Weather
df['Weather.Condition'] = df['Weather.Condition'].replace({'UNK': np.nan, 'Unk': np.nan, 'Unknown': np.nan})

# Number of engines
df['Number.of.Engines'] = df['Number.of.Engines'].where(df['Number.of.Engines'].between(1, 4))

# Purpose of flight
pof = df['Purpose.of.flight'].replace({'Unknown': np.nan})
pof = pof.where(~pof.str.startswith('Public Aircraft', na=False), 'Public Aircraft')
small_cats = pof.value_counts(normalize=True)
small_cats = small_cats[small_cats < 0.01].index
df['Purpose.of.flight'] = pof.replace({s: 'Other' for s in small_cats})

# Phase of flight
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].replace({'Unknown': np.nan, 'Other': np.nan})

for c in ['Engine.Type', 'Weather.Condition', 'Number.of.Engines', 'Purpose.of.flight', 'Broad.phase.of.flight']:
    print('-----', c)
    print(df[c].value_counts(dropna=False))

----- Engine.Type
Engine.Type
Reciprocating    49183
NaN               3295
Turbo Prop        2217
Turbo Fan          887
Turbo Jet          202
Other               14
Name: count, dtype: int64
----- Weather.Condition
Weather.Condition
VMC    49082
IMC     4354
NaN     2362
Name: count, dtype: int64
----- Number.of.Engines
Number.of.Engines
1.0    46375
2.0     6948
NaN     2228
3.0      140
4.0      107
Name: count, dtype: int64
----- Purpose.of.flight
Purpose.of.flight
Personal              33204
Instructional          7834
NaN                    5634
Aerial Application     3155
Business               2676
Other                  2402
Positioning             893
Name: count, dtype: int64
----- Broad.phase.of.flight
Broad.phase.of.flight
NaN            16214
Landing        11090
Takeoff         8185
Cruise          6585
Maneuvering     4250
Approach        4168
Taxi            1323
Climb           1282
Descent         1161
Go-around       1101
Standing         439
Name: count, dtype: i

## 7. Column removal
Rule from the brief: **keep columns with more than 20,000 non-null values.** We apply that rule first, then also drop identifier/location columns that are not relevant to the client's question.

In [17]:
nn = df.notna().sum().sort_values(ascending=False)
print(nn)
df = df[nn[nn > 20000].index].copy()
print('\nColumns kept by the 20,000 rule:', list(df.columns))

Event.Id                  55798
Investigation.Type        55798
Accident.Number           55798
Event.Date                55798
Aircraft.Category         55798
Aircraft.damage           55798
plane_type                55798
_model_key                55798
Injury.Severity           55798
Make                      55798
Model                     55798
Amateur.Built             55798
fatal_serious_frac        55798
destroyed                 55798
_make_key                 55798
Total.Minor.Injuries      55798
total_passengers          55798
fatal_serious             55798
Total.Uninjured           55798
Total.Fatal.Injuries      55798
Total.Serious.Injuries    55798
Location                  55776
Country                   55720
Registration.Number       55264
Number.of.Engines         53570
Weather.Condition         53436
Report.Status             52735
Engine.Type               52503
Purpose.of.flight         50164
Publication.Date          46450
Broad.phase.of.flight     39584
Airport.

In [18]:
# Drop helper keys and columns that don't inform safety outcomes (IDs, free text, geo-coordinates, airport info)
drop_cols = ['Investigation.Type', 'Accident.Number', 'Aircraft.Category', 'Amateur.Built', '_make_key', '_model_key',
             'Injury.Severity', 'Location', 'Registration.Number', 'Report.Status', 'Publication.Date',
             'Airport.Name', 'Airport.Code', 'Latitude', 'Longitude']
df = df.drop(columns=[c for c in drop_cols if c in df.columns]).reset_index(drop=True)
print(df.shape)
df.isna().sum()

(55798, 20)


Event.Id                      0
Event.Date                    0
Aircraft.damage               0
plane_type                    0
Make                          0
Model                         0
fatal_serious_frac            0
destroyed                     0
Total.Minor.Injuries          0
total_passengers              0
fatal_serious                 0
Total.Uninjured               0
Total.Fatal.Injuries          0
Total.Serious.Injuries        0
Country                      78
Number.of.Engines          2228
Weather.Condition          2362
Engine.Type                3295
Purpose.of.flight          5634
Broad.phase.of.flight     16214
dtype: int64

## 8. Save cleaned data

In [19]:
df.to_csv('data/cleaned_aviation.csv', index=False)
df.head()

,Event.Id,Event.Date,Aircraft.damage,plane_type,Make,Model,fatal_serious_frac,destroyed,Total.Minor.Injuries,total_passengers,fatal_serious,Total.Uninjured,Total.Fatal.Injuries,Total.Serious.Injuries,Country,Number.of.Engines,Weather.Condition,Engine.Type,Purpose.of.flight,Broad.phase.of.flight
0,20001214X42095,1983-01-01,Substantial,CESSNA 182P,CESSNA,182P,0.0,0,1.0,4.0,0.0,3.0,0.0,0.0,United States,1.0,VMC,Reciprocating,Personal,Approach
1,20001214X42067,1983-01-01,Substantial,CESSNA 182RG,CESSNA,182RG,0.0,0,0.0,2.0,0.0,2.0,0.0,0.0,United States,1.0,VMC,Reciprocating,Personal,Landing
2,20001214X42063,1983-01-01,Substantial,CESSNA 182P,CESSNA,182P,0.0,0,0.0,1.0,0.0,1.0,0.0,0.0,United States,1.0,VMC,Reciprocating,Personal,Takeoff
3,20001214X42018,1983-01-01,Substantial,PIPER PA28R200,PIPER,PA28R200,0.0,0,2.0,2.0,0.0,0.0,0.0,0.0,United States,1.0,VMC,Reciprocating,Personal,Approach
4,20001214X41951,1983-01-01,Substantial,CESSNA 140,CESSNA,140,0.0,0,0.0,2.0,0.0,2.0,0.0,0.0,United States,1.0,VMC,Reciprocating,Instructional,Landing
